# pyPetrograph — AMICS workflow (v0.0.7)

One field, three images: a BSE photo, a painted mineral map, and a legend image. The notebook reads mineral names from the legend, outlines grains on the BSE, names each grain from the map, and writes counts, size curves, and a knolling sheet (one band per mineral, long axis up, largest ECD first). Grain size is the a-axis (long), b-axis (short), D = √(a·b), and ECD = √(4A/π) with A the projected area.

Run the cells **in order**. File, threshold, and scale windows need a screen (this computer, not Colab). With no screen, set the paths, `BSE_THRESHOLD`, and `UM_PER_PX` in Settings instead. The grain editor (GrainPlot) opens after segmentation, and again on saved grains; close its window to save. Results go to `outputs/amics/{sample}/`.

| Setting | What it does |
|---|---|
| `FIG_DPI = 150` | Saved figure resolution. |
| `START_DIR` | Folder the file window opens in. |
| `LEGEND_PATH`, `BSE_PATH`, `MINERAL_PATH` | `None` opens one file window with three labeled rows, filled with the paths you picked last time. Picking the BSE image or the map fills the other rows by sample name. Set all three paths to skip the window. |
| `SAMPLE` | `None` takes the text before the first `_` in the BSE file name (`Con-1_BSE_Image.tiff` → `Con-1`). |
| `SKIP_NAMES` | These legend names are not a grain's mineral. A grain made only of them is Unknown. |
| `COLOR_TOL = 10` | A map pixel belongs to a legend color within this RGB distance. |
| `WHITE_TOL = 3` | Near-white map pixels are background. Pores (250, 250, 250) still count. |
| `AUTO_MATCH_COLORS = True` | Map colors that match a legend swatch exactly keep that name. The other map colors are paired with the legend names that are still free, one color per name, using the best fit over the whole set. A colored map color left over takes its nearest mineral name, even one already used. Only grays with no gray legend entry stay Unknown. The list is printed before counting. `False` uses only the legend swatch values. |
| `MATCH_MAX_DIST = 25` | Largest color difference (CIEDE2000; about 2 is just visible) for pairing with a free name. Beyond it, a colored map color shares its nearest mineral name instead. |
| `FOOTER_ROW` | `None` finds the red scale-bar strip. Set a row to override. |
| `THR_RADIUS = 5`, `THR_PCT = 99` | Clicks on epoxy or holes: gray level is the 99th percentile in a 5 px radius, plus 1. |
| `BSE_THRESHOLD` | `None` always opens the threshold window, starting from your saved points. A number (0–255 on the contrast-stretched BSE) skips it. |
| `SCALE_LENGTH = 500`, `SCALE_UNIT = "µm"` | Defaults in the scale dialog. The line you draw is the real length. |
| `UM_PER_PX` | `None` always opens the scale window, showing the saved line. A number (µm per pixel) skips it. |
| `MIN_GRAIN_PX = 10` | Size curves and the count `n` use grains at least this big. Half this size is kept so the low count range has grains. |
| `DARK_DROP_FRAC = 0.5` | After segmentation, grains that are more than half dark (epoxy or holes) are dropped. Dark pixels are cut out of the rest. |
| `USE_SAM2 = True` | SAM 2.1 refines outlines. First use downloads about 860 MB and is slow. `False` is the U-Net only. |
| `EDIT_GRAINS = True` | Always opens GrainPlot (also on saved grains) so you can add, delete, or merge grains. `False` saves without a window. |
| `UNMAPPED_FRAC = 0.5` | A grain with more than this share of its pixels on blank (white) map is Unmapped: dropped from counts, CSVs, and knolling, kept in `minerals.geojson`. |
| `MIXED_FRAC = 0.30` | A grain is mixed when the second mineral covers at least this share. Mixed grains are circled. |
| `OVERLAY_ALPHA = 0.5` | Mineral-map tint on the BSE. |
| `KNOLL_ALPHA = 0.5`, `KNOLL_WIDTH_PX` | Tint on knolling tiles. `None` makes a 5500 px wide sheet (grains drawn at 2×, same shape) that grows taller. Set a pixel width to override. |
| `MIN_GRAINS_CURVE = 5` | A mineral needs this many grains to get its own size figure (`{sample}_gsd_{mineral}.png`). |
| `LOAD_SAVED_FILES` | The file window starts with the paths you picked last time. `False` starts empty. |
| `LOAD_SAVED_THRESHOLD`, `LOAD_SAVED_SCALE` | The threshold and scale windows start from the last points / line. `False` starts blank. |
| `LOAD_SAVED_GRAINS` | Start from the saved grain file (GrainPlot opens on it when `EDIT_GRAINS`). `False` runs segmentation again. |


## Setup


In [ ]:
%matplotlib inline

from pathlib import Path
import json
import sys

ROOT = Path(".")
if str(ROOT.resolve()) not in sys.path:
    sys.path.insert(0, str(ROOT.resolve()))

FIG_DPI = 150

from pyPetrograph import check_and_install_packages
_ = check_and_install_packages(
    install=True, include_seg=True, include_sam2=True, include_amics=True,
    include_app=True,  # reports the desktop app packages (part of pip install pyPetrograph); never installs them
)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection
from PIL import Image

plt.rcParams["figure.dpi"] = FIG_DPI
plt.rcParams["savefig.dpi"] = FIG_DPI

from pyPetrograph import set_output_dir
from pyPetrograph.amics import (
    MEASURE_COLUMNS,
    add_size_metrics,
    amics_paths,
    click_points,
    count_table,
    dark_mask,
    detect_footer,
    draw_scale,
    legend_to_text,
    load_bse,
    load_mineral_map,
    make_knolling,
    match_map_colors,
    measure_grains,
    mineral_gsd_png,
    pick_amics_files,
    plot_gsd,
    plot_gsd_by_mineral,
    plot_legend,
    plot_missing_colors,
    plot_composition,
    plot_mixed,
    plot_overlay,
    plot_threshold_preview,
    plot_unmapped,
    read_footer_pixelsize,
    read_grains_geojson,
    read_legend,
    recolor_from_matches,
    run_seg_bse,
    sample_name,
    save_figure,
    save_knolling,
    stretch,
    threshold_from_points,
    valid_mask,
    vote_grains,
    write_grains_geojson,
    write_mineral_csvs,
)


## Settings


In [ ]:
set_output_dir("outputs")

FIG_DPI = 150
plt.rcParams["figure.dpi"] = FIG_DPI
plt.rcParams["savefig.dpi"] = FIG_DPI

START_DIR = Path("example_data")
LEGEND_PATH = None  # or Path("example_data/Con-1_Mineral_Legend.tiff")
BSE_PATH = None     # or Path("example_data/Con-1_BSE_Image.tiff")
MINERAL_PATH = None # or Path("example_data/Con-1_Mineral_Image.tiff")
# any None opens the file window (prefilled with the ones you set)
SAMPLE = None

SKIP_NAMES = ["Unknown", "Low_Counts", "Un_x_rayed", "Shadows", "Pores"]
COLOR_TOL = 10
WHITE_TOL = 3
AUTO_MATCH_COLORS = True
MATCH_MAX_DIST = 25
FOOTER_ROW = None

THR_RADIUS = 5
THR_PCT = 99
BSE_THRESHOLD = None

SCALE_LENGTH = 500
SCALE_UNIT = "µm"
UM_PER_PX = None  # a number (µm per pixel) skips the scale window

MIN_GRAIN_PX = 10
DARK_DROP_FRAC = 0.5
USE_SAM2 = True
EDIT_GRAINS = True
UNMAPPED_FRAC = 0.5
MIXED_FRAC = 0.30
OVERLAY_ALPHA = 0.5
KNOLL_ALPHA = 0.5
KNOLL_WIDTH_PX = None
MIN_GRAINS_CURVE = 5

LOAD_SAVED_FILES = True
LOAD_SAVED_THRESHOLD = True
LOAD_SAVED_SCALE = True
LOAD_SAVED_GRAINS = True

print("output folder: outputs/amics/{sample}/")
print(f"SAM 2.1 {'on' if USE_SAM2 else 'off'}; GrainPlot {'on' if EDIT_GRAINS else 'off'}")


## Files

One window with three labeled rows: **Mineral legend**, **BSE image**, **Mineral map**. Paths you set in Settings are already filled in, and so are the paths you picked last time (`LOAD_SAVED_FILES`). If all three Settings paths exist, the window is skipped. Picking the BSE image or the map fills the other rows by sample name. Without a screen, set `LEGEND_PATH`, `BSE_PATH`, `MINERAL_PATH` above.

In [ ]:
FILES = pick_amics_files(LEGEND_PATH, BSE_PATH, MINERAL_PATH, START_DIR, use_saved=LOAD_SAVED_FILES)
LEGEND_FILE, BSE_FILE, MINERAL_FILE = FILES["legend"], FILES["bse"], FILES["mineral"]
for _k, _v in FILES.items():
    print(f"{_k:8s} {_v}")

## 1. Legend

The legend image comes from the file window above. Names come from the text beside each color swatch. Shared colors become one class, for example `"Sanidine / Orthoclase"`. Check the printed names against the image. OCR drops some `_1` suffixes and can misread a letter.


In [ ]:
LEGEND_AUTO, LEGEND_REPORT = read_legend(LEGEND_FILE)
legend_img = np.asarray(Image.open(LEGEND_FILE))
fig, ax = plt.subplots(figsize=(8, max(3, legend_img.shape[0] / 80)))
ax.imshow(legend_img)
ax.set_title(LEGEND_FILE.name)
ax.axis("off")
plt.show()

print("\n".join(LEGEND_REPORT))
print()
print("Paste the dict below into LEGEND_EDIT if a name is wrong. Leave LEGEND_EDIT = None to keep it.")
print(legend_to_text(LEGEND_AUTO))


Edit names here if the printout is wrong. Example: `LEGEND_EDIT = {"Quartz": (255, 255, 0), "Unknown": (0, 0, 0)}`. Leave `None` to keep the read legend.


In [ ]:
LEGEND_EDIT = None

LEGEND = dict(LEGEND_EDIT) if LEGEND_EDIT else dict(LEGEND_AUTO)
fig = plot_legend(LEGEND, SKIP_NAMES)
plt.show()
print(f"{len(LEGEND)} legend entries")


## 2. BSE grains

The BSE image comes from the file window. The sample name and the output folder are set here, and the legend figure is saved. The red dashed line is the top of the scale-bar strip. Rows below it are not grains.


In [ ]:
SAMPLE = SAMPLE or sample_name(BSE_FILE)
P = amics_paths(SAMPLE, BSE_FILE)

P["legend_json"].write_text(
    json.dumps({name: [int(c) for c in rgb] for name, rgb in LEGEND.items()}, indent=1),
    encoding="utf-8",
)
fig = plot_legend(LEGEND, SKIP_NAMES)
save_figure(fig, P["legend_png"], FIG_DPI)
plt.close(fig)

BSE_GRAY, BSE_RGBA = load_bse(BSE_FILE)
FOOTER = int(FOOTER_ROW) if FOOTER_ROW is not None else int(detect_footer(BSE_RGBA))
BSE_VALID = valid_mask(BSE_RGBA, FOOTER)
BSE_STRETCH = stretch(BSE_GRAY, BSE_VALID)

print(f"sample {SAMPLE}")
print(f"image {BSE_GRAY.shape[1]} x {BSE_GRAY.shape[0]} px; data rows end at {FOOTER}")
print("files:", P["folder"])

h, w = BSE_GRAY.shape[:2]
fig, ax = plt.subplots(figsize=(10, 10 * h / w))
ax.imshow(BSE_GRAY, cmap="gray", vmin=0, vmax=255)
if FOOTER < h:
    ax.axhline(FOOTER - 0.5, color="red", linestyle="--", linewidth=1)
ax.set_title(f"{BSE_FILE.name}  footer row {FOOTER}")
ax.axis("off")
plt.show()


### Threshold

This window always opens (unless `BSE_THRESHOLD` is a number) and starts from your saved points. Left click adds a point on epoxy or a hole, right click removes the nearest point, C clears all, Enter or closing the window finishes. The blue tint and the threshold number update after every click. Darker than that level, plus the footer and empty mosaic gaps, is removed from the grain outlines.

In [ ]:
saved_thr = None
if LOAD_SAVED_THRESHOLD and P["threshold_json"].exists():
    saved_thr = json.loads(P["threshold_json"].read_text(encoding="utf-8"))
SAVED_POINTS = [tuple(p) for p in ((saved_thr or {}).get("points") or [])]

POINTS = []
if BSE_THRESHOLD is not None:
    THRESHOLD = float(BSE_THRESHOLD)
    print(f"using BSE_THRESHOLD = {THRESHOLD}")
else:
    print("Threshold window: left click adds | right click removes nearest | C clears all | Enter or close = done.")
    POINTS = click_points(
        BSE_STRETCH, BSE_VALID, SAVED_POINTS, radius=THR_RADIUS, pct=THR_PCT, footer_row=FOOTER
    )
    if not POINTS:
        raise RuntimeError("No points. Click at least one epoxy / hole pixel, or set BSE_THRESHOLD in Settings.")
    THRESHOLD = threshold_from_points(
        BSE_STRETCH, POINTS, BSE_VALID, radius=THR_RADIUS, pct=THR_PCT
    )

DARK = dark_mask(BSE_STRETCH, THRESHOLD, BSE_VALID)
P["threshold_json"].write_text(
    json.dumps({"threshold": THRESHOLD, "points": [[float(x), float(y)] for x, y in POINTS]}, indent=1),
    encoding="utf-8",
)
fig = plot_threshold_preview(BSE_STRETCH, DARK, FOOTER, POINTS)
save_figure(fig, P["threshold_png"], FIG_DPI)
plt.show()
print(f"threshold {THRESHOLD:.1f}; dark or invalid {100 * float(DARK.mean()):.1f}% of the image")

### Scale

This window always opens (unless `UM_PER_PX` is a number). The saved line is drawn in orange. Click two new points to draw another, Enter to type its length (or just Enter to change the length of the saved line). Closing the window without Enter keeps the current scale. Footer text such as `PixelSize: 2.7µm` is only a cross-check.

In [ ]:
SAVED_SCALE = None
if LOAD_SAVED_SCALE and P["scale_json"].exists():
    SAVED_SCALE = json.loads(P["scale_json"].read_text(encoding="utf-8"))

if UM_PER_PX is not None:
    SCALE = {"um_per_px": float(UM_PER_PX)}
    print(f"using UM_PER_PX = {UM_PER_PX}")
else:
    print("Scale window: click 2 points to draw a new line | Enter = type length | close = keep current scale.")
    NEW_SCALE = draw_scale(BSE_GRAY, saved=SAVED_SCALE, default_length=SCALE_LENGTH, default_unit=SCALE_UNIT)
    if NEW_SCALE is not None:
        SCALE = NEW_SCALE
        P["scale_json"].write_text(json.dumps(SCALE, indent=1), encoding="utf-8")
    elif SAVED_SCALE:
        SCALE = SAVED_SCALE
        print(f"kept the saved scale from {P['scale_json'].name}")
    else:
        print("No scale. Lengths stay in pixels.")
        SCALE = {}

UM_PER_PX_USED = SCALE.get("um_per_px")
SIZE_UNIT = "um" if UM_PER_PX_USED else "px"
if UM_PER_PX_USED and SCALE.get("px_length"):
    print(f"scale {SCALE.get('length')} {SCALE.get('unit')} = {SCALE.get('px_length')} px -> {float(UM_PER_PX_USED):.4f} µm/px")
elif UM_PER_PX_USED:
    print(f"scale {float(UM_PER_PX_USED):.4f} µm/px")
else:
    print("scale: none")

ocr_um = read_footer_pixelsize(BSE_RGBA, FOOTER)
print(f"footer text cross-check: {ocr_um} µm/px" if ocr_um else "footer text cross-check: not read")

### Grain outlines

SegmentEveryGrain runs in a separate process (the kernel does not load TensorFlow). With `EDIT_GRAINS`, a GrainPlot window always opens, on the new outlines or on your saved ones (`LOAD_SAVED_GRAINS`). The top of the window lists the keys: click a grain to select it, D deletes, M merges the selected grains, Z undoes the last add, Esc clears; with SAM 2.1 on, clicking an empty area adds a grain. Close the window to save.

In [ ]:
def _rings(geom):
    if geom is None or getattr(geom, "is_empty", True):
        return
    if geom.geom_type == "Polygon":
        yield np.asarray(geom.exterior.coords)
        return
    for part in getattr(geom, "geoms", ()):
        yield from _rings(part)


if LOAD_SAVED_GRAINS and P["grains"].exists():
    if EDIT_GRAINS:
        print(f"Opening GrainPlot on the saved grains in {P['grains'].name}.")
        GRAIN_POLYS = run_seg_bse(
            BSE_STRETCH,
            DARK,
            P["grains"],
            use_sam=USE_SAM2,
            edit=True,
            load_from=P["grains"],
        )
    else:
        GRAIN_POLYS, _saved, _meta = read_grains_geojson(P["grains"])
    print(f"{len(GRAIN_POLYS)} grains from {P['grains'].name}")
else:
    note = " GrainPlot opens when outlines are ready; close it to save." if EDIT_GRAINS else ""
    print("Running grain segmentation." + note)
    GRAIN_POLYS = run_seg_bse(
        BSE_STRETCH,
        DARK,
        P["grains"],
        use_sam=USE_SAM2,
        edit=EDIT_GRAINS,
        min_area=MIN_GRAIN_PX / 2,
        drop_frac=DARK_DROP_FRAC,
    )
    print(f"segmentation returned {len(GRAIN_POLYS)} grains")

GRAINS = measure_grains(GRAIN_POLYS, um_per_px=UM_PER_PX_USED)
keep = GRAINS["area_px"].fillna(0) >= (MIN_GRAIN_PX / 2)
GRAIN_POLYS = [poly for poly, ok in zip(GRAIN_POLYS, keep) if ok]
GRAINS = GRAINS.loc[keep].reset_index(drop=True)
GRAINS["grain_id"] = np.arange(1, len(GRAINS) + 1)
write_grains_geojson(
    P["grains"],
    GRAIN_POLYS,
    GRAINS,
    {
        "sample": SAMPLE,
        "um_per_px": UM_PER_PX_USED,
        "scale": SCALE,
        "threshold": THRESHOLD,
        "source_image": str(Path(BSE_FILE)),
    },
)
print(f"kept {len(GRAIN_POLYS)} grains with area >= {MIN_GRAIN_PX / 2:g} px -> {P['grains'].name}")

_h, _w = BSE_STRETCH.shape[:2]
_rows = min(FOOTER, _h)
fig, ax = plt.subplots(figsize=(10, 10 * _rows / _w))
ax.imshow(BSE_STRETCH[:FOOTER], cmap="gray", vmin=0, vmax=255)
segs = [ring for poly in GRAIN_POLYS for ring in _rings(poly)]
if segs:
    ax.add_collection(LineCollection(segs, colors="yellow", linewidths=0.4))
ax.set_xlim(-0.5, _w - 0.5)
ax.set_ylim(_rows - 0.5, -0.5)
ax.axis("off")
ax.set_title(f"{SAMPLE}: {len(GRAIN_POLYS)} grains")
plt.show()


### Size metrics

Each grain gets four sizes: **a** (long axis), **b** (short axis), **D** = √(a·b), and **ECD** = √(4A/π), where A is the projected area. Each is in pixels, and in µm when a scale exists. The grain file is rewritten with all four, and so are the later CSVs.


In [ ]:
GRAINS = add_size_metrics(GRAINS, UM_PER_PX_USED)
write_grains_geojson(
    P["grains"],
    GRAIN_POLYS,
    GRAINS,
    {
        "sample": SAMPLE,
        "um_per_px": UM_PER_PX_USED,
        "scale": SCALE,
        "threshold": THRESHOLD,
        "source_image": str(Path(BSE_FILE)),
    },
)
_unit = "µm" if SIZE_UNIT == "um" else "px"
_big = GRAINS["area_px"] >= MIN_GRAIN_PX
print(f"{int(_big.sum())} grains >= {MIN_GRAIN_PX:g} px. Medians ({_unit}):")
print("  " + ", ".join(f"{_n} {GRAINS.loc[_big, f'{_n}_{SIZE_UNIT}'].median():.2f}" for _n in ("a", "b", "D", "ECD")))
print(f"rewrote {P['grains'].name} with a, b, D, ECD")


### Grain size

Curves use grains at least `MIN_GRAIN_PX`. Left column: **D** is the shaded histogram and solid curve, **b** is dashed, **a** is dotted. Right column: **ECD**. Top is the share of grains in each log size bin. Bottom is the share of grains smaller than that size, as a line from 0 to 100%. Bins run from the smallest to the largest grain.


In [ ]:
big = GRAINS["area_px"] >= MIN_GRAIN_PX
fig = plot_gsd(GRAINS.loc[big], unit=SIZE_UNIT)
save_figure(fig, P["gsd"], FIG_DPI)
plt.show()
print(f"n = {int(big.sum())} grains >= {MIN_GRAIN_PX:g} px")


## 3. Minerals

The mineral map (from the file window) must be the same width and height as the BSE. Colors the legend does not name are listed with a patch and a pixel count. Name them in the next cell. Unnamed colors count as Unknown.


In [ ]:
map_sample = sample_name(MINERAL_FILE)
if map_sample != SAMPLE:
    print(f"warning: map file starts with {map_sample!r}, sample is {SAMPLE!r}")

MAP_RGB, CMAP, MAP_VALID, MISSING = load_mineral_map(
    MINERAL_FILE, LEGEND, footer_row=FOOTER, color_tol=COLOR_TOL, white_tol=WHITE_TOL
)
if MAP_RGB.shape[:2] != BSE_GRAY.shape[:2]:
    raise ValueError(
        f"mineral map is {MAP_RGB.shape[1]}x{MAP_RGB.shape[0]}, BSE is {BSE_GRAY.shape[1]}x{BSE_GRAY.shape[0]}"
    )

fig = plot_missing_colors(MISSING)
plt.show()
if len(MISSING):
    print(MISSING.head(15).to_string(index=False))
    print("Name these in EXTRA_LEGEND (next cell), then run that cell. Unnamed colors count as Unknown.")
else:
    print("Every map color is in the legend.")


Add map colors the legend missed. Keys are the names you want. Values are the `(r, g, b)` from the table above. Example: `EXTRA_LEGEND = {"Chromite": (255, 0, 255)}`. These are applied first. With `AUTO_MATCH_COLORS`, the colors still unnamed are then paired with the legend names no exact color uses, and the list is printed below.


In [ ]:
EXTRA_LEGEND = {}

VOTE_LEGEND = {**LEGEND, **EXTRA_LEGEND}
MAP_RGB, CMAP, MAP_VALID, MISSING = load_mineral_map(
    MINERAL_FILE, VOTE_LEGEND, footer_row=FOOTER, color_tol=COLOR_TOL, white_tol=WHITE_TOL
)

COLOR_MATCHES = None
if AUTO_MATCH_COLORS and len(MISSING):
    COLOR_MATCHES = match_map_colors(MISSING, VOTE_LEGEND, CMAP, SKIP_NAMES, max_dist=MATCH_MAX_DIST)
    print("Map colors with no exact legend swatch:")
    for _, _m in COLOR_MATCHES.iterrows():
        _c = f"({_m.map_red}, {_m.map_green}, {_m.map_blue})"
        if _m.status == "free name":
            _how = f"{_m.assigned}  (unused legend name, color difference {_m.distance:.0f})"
        elif _m.status == "shared name":
            _how = f"{_m.assigned}  (nearest mineral, name also used by its exact color; difference {_m.distance:.0f})"
        else:
            _how = "left unnamed (gray, no gray legend entry): counts as Unknown"
        print(f"  {_c:18s} {_m.pixels:>9,d} px -> {_how}")
    _recolor = recolor_from_matches(COLOR_MATCHES, VOTE_LEGEND)
    if _recolor:
        MAP_RGB, CMAP, MAP_VALID, MISSING = load_mineral_map(
            MINERAL_FILE, VOTE_LEGEND, footer_row=FOOTER, color_tol=COLOR_TOL, white_tol=WHITE_TOL, recolor=_recolor
        )
print(f"vote legend: {len(VOTE_LEGEND)} names; colors still unnamed: {len(MISSING)}")


BSE underneath, mineral colors at half strength, grain outlines on top.


In [ ]:
fig = plot_overlay(
    BSE_STRETCH[:FOOTER], MAP_RGB[:FOOTER], MAP_VALID[:FOOTER], GRAIN_POLYS, alpha=OVERLAY_ALPHA
)
save_figure(fig, P["overlay"], FIG_DPI)
plt.show()


Each grain is named by the mineral that covers the most of its pixels. Skipped names (epoxy-like legend entries) do not vote. A red circle marks a mixed grain. A grain with more than `UNMAPPED_FRAC` of its pixels on blank (white) map has no mineral data; it is called Unmapped and dropped from the counts, CSVs, knolling, and per-mineral curves. The cyan outlines in the first figure show which grains those are.

In [ ]:
votes = vote_grains(
    GRAIN_POLYS, CMAP, VOTE_LEGEND, SKIP_NAMES, mixed_frac=MIXED_FRAC, unmapped_frac=UNMAPPED_FRAC
)
base_cols = [c for c in MEASURE_COLUMNS if c in GRAINS.columns]
GRAINS = pd.concat(
    [GRAINS.reindex(columns=base_cols).reset_index(drop=True), votes.drop(columns=["grain_id"]).reset_index(drop=True)],
    axis=1,
)

fig = plot_unmapped(
    BSE_STRETCH[:FOOTER], MAP_RGB[:FOOTER], MAP_VALID[:FOOTER], GRAIN_POLYS, GRAINS, alpha=OVERLAY_ALPHA
)
save_figure(fig, P["unmapped"], FIG_DPI)
plt.show()

write_grains_geojson(
    P["minerals"],
    GRAIN_POLYS,
    GRAINS,
    {"sample": SAMPLE, "um_per_px": UM_PER_PX_USED, "source_image": str(Path(MINERAL_FILE))},
)
print(f"wrote {P['minerals'].name} (all {len(GRAINS)} grains, Unmapped included)")

is_mapped = (GRAINS["mineral"] != "Unmapped").to_numpy()
MAPPED = GRAINS.loc[is_mapped].reset_index(drop=True)
MAPPED_POLYS = [poly for poly, ok in zip(GRAIN_POLYS, is_mapped) if ok]
print(f"dropped {int((~is_mapped).sum())} unmapped grains (no map color under them); {len(MAPPED)} grains kept")

fig = plot_mixed(
    BSE_STRETCH[:FOOTER], MAP_RGB[:FOOTER], MAP_VALID[:FOOTER], MAPPED_POLYS, MAPPED, alpha=OVERLAY_ALPHA
)
save_figure(fig, P["mixed"], FIG_DPI)
plt.show()
print(MAPPED["mineral"].value_counts().to_string())

COUNTS = count_table(
    MAPPED, VOTE_LEGEND, SKIP_NAMES, sample=SAMPLE, min_grain_px=MIN_GRAIN_PX, unit=SIZE_UNIT
)
COUNTS

### Composition

Left: each counted grain (at least `MIN_GRAIN_PX`) painted in its mineral color. Right: the percent of those grains in each mineral, in the same colors. The table under the figure is the count and that percent.


In [ ]:
_keep = (MAPPED["area_px"] >= MIN_GRAIN_PX).to_numpy()
fig = plot_composition(
    [poly for poly, ok in zip(MAPPED_POLYS, _keep) if ok],
    MAPPED.loc[_keep].reset_index(drop=True),
    VOTE_LEGEND,
    COUNTS,
    sample=SAMPLE,
    shape=BSE_STRETCH[:FOOTER].shape[:2],
)
save_figure(fig, P["composition"], FIG_DPI)
plt.show()
_share = COUNTS.loc[COUNTS["n"] > 0, ["class", "n", "share_pct"]].copy()
_share["share_pct"] = _share["share_pct"].round(1)
_share


One size figure per mineral with at least `MIN_GRAINS_CURVE` grains, in the same layout (a, b, D left; ECD right), in the mineral's legend color. Files are `{sample}_gsd_{mineral}.png`. A mineral with few grains rises in big steps.


In [ ]:
big = MAPPED["area_px"] >= MIN_GRAIN_PX
for _mineral, fig in plot_gsd_by_mineral(
    MAPPED.loc[big], unit=SIZE_UNIT, colors=VOTE_LEGEND, min_grains=MIN_GRAINS_CURVE
):
    save_figure(fig, mineral_gsd_png(P["folder"], SAMPLE, _mineral), FIG_DPI)
    plt.show()


## 4. Outputs

One counts table, one CSV per mineral (largest a-axis first), each with a, b, D, ECD in px and µm, and one knolling image. The CSVs and knolling use the counted grains (mapped, at least `MIN_GRAIN_PX`). Knolling keeps relative size (every grain at 2×) on a fixed 5500 px wide sheet, tiles ordered by ECD (largest first), so a long sheet is expected.


In [ ]:
counted = (MAPPED["area_px"] >= MIN_GRAIN_PX).to_numpy()
COUNTED = MAPPED.loc[counted].reset_index(drop=True)
COUNTED_POLYS = [poly for poly, ok in zip(MAPPED_POLYS, counted) if ok]

COUNTS.to_csv(P["counts"], index=False)
csv_paths = write_mineral_csvs(COUNTED, SAMPLE, P["folder"])
print(f"wrote {P['counts'].name}")
print(f"wrote {len(csv_paths)} mineral CSVs in {P['folder']}")
COUNTS


In [ ]:
knoll = make_knolling(
    BSE_STRETCH,
    COUNTED_POLYS,
    COUNTED,
    VOTE_LEGEND,
    alpha=KNOLL_ALPHA,
    width_px=KNOLL_WIDTH_PX,
)
save_knolling(knoll, P["knolling"], FIG_DPI)
fig_w = 40.0
fig_h = fig_w * knoll.shape[0] / max(knoll.shape[1], 1)
fig, ax = plt.subplots(figsize=(fig_w, fig_h))
ax.imshow(knoll)
ax.set_title(P["knolling"].name)
ax.axis("off")
plt.show()
print(f"knolling {knoll.shape[1]} x {knoll.shape[0]} px -> {P['knolling'].name}")
